# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/neelamobaid/flyrank-ml-internship/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

*Name your lane — or say 'freestyle' and describe your own question. One short paragraph: why this one?*

## 1. My lane (or freestyle) and why

**Lane: Lane 2, Refresh / Content Opportunity Scoring**

**Question:** Which pages should a content team review first? I want to test whether combining performance signals (content age, time since last update, impressions, clicks, engagement, trend) produces a better review queue than random selection or a single rule.

**Why this lane:** A content team cannot review every page, so a ranked list of pages worth a look is a practical way to use the data. The dataset has the staleness, search exposure and trend columns this lane needs.

**My angle:** I will evaluate the review queue under a fixed monthly capacity (the top K pages), since a team can only act on a limited number of pages. A ranking is only useful if the pages at the top are the right ones.

**Unit and output:** One content page; a ranked review list with the reasons each page was flagged (for example: stale, high impressions, down trend).

This choice is provisional, and I can confirm or change it by the end of Week 4.

## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

## 2. The question: decision, action, cost of a wrong call

**Decision:** Which pages a content team reviews first, when it can only review K pages per month.

**Who acts and what they do:** An editor or SEO manager reads the ranked list, then decides whether to update, monitor, or leave each page unchanged. This output only supports their decision; it does not change anything automatically.

**Cost of a wrong call:**
- *False positive:* a page is flagged unnecessarily and limited editor time is wasted.
- *False negative:* a page with a real problem and meaningful search exposure goes unreviewed.

**Why data analysis, and why not "just train a model":** With thousands of pages and several signals, a single rule may miss useful pages. I will compare random selection, single rules, a hand-weighted combined score, and possibly a model later. If a simple rule performs as well as a model, that is a valid finding, and a model would not be justified.

**Caveat on "needs review":** The dataset has no ground-truth label for this. Any target I use will be a proxy, and I will say so.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

In [2]:
import os
import pandas as pd

if not os.path.exists("flyrank-ml-internship") and not os.path.exists("data"):
    !git clone https://github.com/neelamobaid/flyrank-ml-internship.git

candidates = [
    "data/raw/content_refresh_anonymized.csv",
    "../../data/raw/content_refresh_anonymized.csv",
    "flyrank-ml-internship/data/raw/content_refresh_anonymized.csv",
    "/content/flyrank-ml-internship/data/raw/content_refresh_anonymized.csv",
]
path = next(p for p in candidates if os.path.exists(p))
df = pd.read_csv(path)

print("Total pages:", len(df))
print("Clients:", df["client_id"].nunique())
print("Share of pages with a 'down' trend label:", round((df["trend_direction"] == "down").mean(), 3))

Total pages: 30000
Clients: 32
Share of pages with a 'down' trend label: 0.542


In [3]:
# 1) Is search exposure concentrated? (does prioritising matter?)
imp = df["impressions_90d"].sort_values(ascending=False)
top10 = imp.head(int(len(df) * 0.10)).sum() / imp.sum()
print(f"Top 10% of pages hold {top10:.1%} of impressions")

# 2) How much exposure sits on 'down' pages?
down = df[df["trend_direction"] == "down"]
print(f"'down' pages hold {down['impressions_90d'].sum() / df['impressions_90d'].sum():.1%} of impressions")

# 3) Is staleness related to trend at all?
print(df.groupby("trend_direction")["days_since_last_update"].median())

Top 10% of pages hold 70.2% of impressions
'down' pages hold 51.3% of impressions
trend_direction
down      20.0
flat      20.0
new       20.0
stable    22.0
up        22.0
Name: days_since_last_update, dtype: float64


**What these numbers show (observed, not causal):**

- About 54% of pages carry a "down" label, so the trend label alone does not narrow the list much. Ranking within it is the real problem.
- The top 10% of pages hold 70.2% of impressions, which suggests that a top-K ranking can matter: a small share of pages accounts for most search exposure.
- Pages labelled "down" hold 51.3% of impressions, close to their share of pages, so the label does not single out high-exposure pages.
- Median days since last update is 20 for down pages and 22 for up and stable pages. This is a small difference, so staleness may carry little signal on its own. I will test this more carefully before relying on it in a ranking.

## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*
**What I can claim:** Observed patterns in this dataset, and a comparison of ranking approaches under a fixed review budget. The results are decision-support for choosing which pages to look at first.

**What I cannot claim:**
- That a "down" label proves a page will keep declining or needs editing.
- That updating a page will improve traffic or engagement. The data has no record of edits or their effects.
- Anything about predicting Google's ranking decisions.

**Known limitation:** Trend is both a candidate input signal and the label I looked at above. Before evaluating rankings, I will choose a target that is not also an input (for example, impressions at stake in the top K), so the results are not circular.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.